In [129]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression


from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings('ignore')

In [130]:
df = pd.read_csv("heart.csv")
print(df.shape)
print(df.isna().sum())
df.head(3)

(918, 12)
Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0


In [131]:
df.columns.tolist()

['Age',
 'Sex',
 'ChestPainType',
 'RestingBP',
 'Cholesterol',
 'FastingBS',
 'RestingECG',
 'MaxHR',
 'ExerciseAngina',
 'Oldpeak',
 'ST_Slope',
 'HeartDisease']

In [132]:
# encoding categorical data
df_encoded = pd.get_dummies(df, drop_first=True)

X = df_encoded.drop('HeartDisease', axis=1)
y = df_encoded['HeartDisease']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)


In [133]:
rf = RandomForestClassifier()
gb = GradientBoostingClassifier()
svc = SVC()
lr = LogisticRegression()

# training random forest
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print(f"RANDOM FOREST     : {np.round(accuracy_score(y_test, y_pred), 2)}")

# training gradient boosting
gb.fit(X_train, y_train)
y_pred2 = gb.predict(X_test)
print(f"GRADIENT BOOSITNG : {np.round(accuracy_score(y_test, y_pred2), 2)}")

# training SVC
svc.fit(X_train, y_train)
y_pred3 = svc.predict(X_test)
print(f"SVC               : {np.round(accuracy_score(y_test, y_pred3), 2)}")

# training logistic regression
lr.fit(X_train, y_train)
y_pred4 = lr.predict(X_test)
print(f"LOGISTIC REGRESSION : {np.round(accuracy_score(y_test, y_pred4), 2)}")

RANDOM FOREST     : 0.88
GRADIENT BOOSITNG : 0.86
SVC               : 0.68
LOGISTIC REGRESSION : 0.85


In [134]:
# using cross_val_score

print(f"RANDOM FOREST : {np.round(np.mean(cross_val_score(rf, X, y, cv=10, scoring='accuracy')), 2)}")
print(f"GRADIENT BOOSTING : {np.round(np.mean(cross_val_score(gb, X, y, cv=10, scoring='accuracy')), 2)}")
print(f"SVC : {np.round(np.mean(cross_val_score(svc, X, y, cv=10, scoring='accuracy')), 2)}")
print(f"LINEAR REGRESSSION : {np.round(np.mean(cross_val_score(lr, X, y, cv=10, scoring='accuracy')), 2)}")


RANDOM FOREST : 0.85
GRADIENT BOOSTING : 0.86
SVC : 0.71
LINEAR REGRESSSION : 0.85


## **Using GridSearchCV** 

In [135]:
param_grid = {
    'n_estimators' : [20, 60, 100, 120],
    'max_features' : [0.2, 0.4, 0.6, 1.0],
    'max_depth' : [2, 4, 8, None],
    'max_samples' : [0.5, 0.25, 0.75, 1.0]
}


print(param_grid['n_estimators'])
print(param_grid['max_features'])
print(param_grid['max_depth'])
print(param_grid['max_samples'])


rf_grid = GridSearchCV(
    estimator = rf,
    param_grid = param_grid,
    cv = 5,
    verbose = 2,
    n_jobs = -1
)

rf_grid.fit(X_train, y_train)

[20, 60, 100, 120]
[0.2, 0.4, 0.6, 1.0]
[2, 4, 8, None]
[0.5, 0.25, 0.75, 1.0]
Fitting 5 folds for each of 256 candidates, totalling 1280 fits


,estimator,RandomForestClassifier()
,param_grid,"{'max_depth': [2, 4, ...], 'max_features': [0.2, 0.4, ...], 'max_samples': [0.5, 0.25, ...], 'n_estimators': [20, 60, ...]}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,20


In [136]:
# will print the best parameters in grid search cv
print(rf_grid.best_params_)

# will print the best score
print(rf_grid.best_score_)


{'max_depth': None, 'max_features': 0.2, 'max_samples': 0.75, 'n_estimators': 20}
0.8787065511135962


## **Using RandomSearchCV**

The difference btwn GridSearchCV and RandomSearchCV is, while using the first one you definitely get the best resualt but it will take a huge lot of time to do the computataions, as it selects the best parameters one by one. And it becomes hell while using it on a very large dataset.

But in case of RandomSearchCV, it finds the best parameters randomly and due to that it takes very less time to do the computatiaons. But one drawback of is, it might not give as accurate resualt as GridSearchCV. 

In [137]:
param_grid2 = {
    "n_estimators" : [20,60,100,120],
    "max_features" : [0.2, 0.4, 0.6,1.0],
    "max_depth" : [2,6,8,None],
    "max_samples" : [0.25,0.5,0.75,1.0],
    "bootstrap" : [True,False],
    "min_samples_split" : [2,3,4,5],
    "min_samples_leaf" : [1,2]
}


rf_grid2 = RandomizedSearchCV(
    estimator = rf,
    param_distributions = param_grid, 
    cv = 5,
    verbose = 2,
    n_jobs = -1
)


rf_grid2.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


,estimator,RandomForestClassifier()
,param_distributions,"{'max_depth': [2, 4, ...], 'max_features': [0.2, 0.4, ...], 'max_samples': [0.5, 0.25, ...], 'n_estimators': [20, 60, ...]}"
,n_iter,10
,scoring,None
,n_jobs,-1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,None
,error_score,nan


In [138]:
print(rf_grid2.best_params_)

print(rf_grid2.best_score_)

{'n_estimators': 120, 'max_samples': 1.0, 'max_features': 0.4, 'max_depth': 4}
0.8623707017053397
